# Horoscope Generation: clean

## imports

In [2]:
import pandas as pd
import ollama
import os
import random
import time
from tqdm import tqdm
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

In [4]:
df = pd.read_pickle("/Users/isishassan/Documents/Ironhack/FinalProject/df_for_text_gen.pkl")

## Model

In [ ]:
MODEL = "llama3.2"
CHECKPOINT_FILE = "horoscopes_ollama.csv"

signs = [
"aries","taurus","gemini","cancer",
"leo","virgo","libra","scorpio",
"sagittarius","capricorn","aquarius","pisces"
]

dates = pd.date_range(start="2025-03-13", end="2026-03-12", freq="D") #end changed to 2025-03-14 for test, change to "2026-03-12"

In [6]:
pip install ollama

Note: you may need to restart the kernel to use updated packages.


## Ollama

In [7]:
# Quick sanity check
import ollama
response = ollama.chat(model="llama3.2", messages=[
    {"role": "user", "content": "Write a 2 sentence horoscope for Aries."}
])
print(response.message.content)

This week, Aries, you're in for a thrilling ride as a new opportunity arises that will challenge you to take risks and push beyond your comfort zone. However, be cautious of your impulsive nature and make sure to weigh the pros and cons carefully, as the pursuit of passion can sometimes lead to reckless decisions.


In [8]:
#testing on 3 rows
import time

prompts = ["Write a 2 sentence horoscope for Aries."] * 3

times = []
for i, p in enumerate(prompts):
    start = time.time()
    response = ollama.chat(model="llama3.2", messages=[{"role": "user", "content": p}])
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"Call {i+1}: {elapsed:.1f}s")

print(f"\nAverage (calls 2-5): {sum(times[1:])/len(times[1:]):.1f}s")

Call 1: 13.1s
Call 2: 2.8s
Call 3: 1.8s

Average (calls 2-5): 2.3s


beware: the below is veeeery slow on a Mac M2 2022, 8GB - runtime was 1800 mins

In [ ]:
import ollama
import os
import random
from tqdm import tqdm
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

MODEL = "llama3.2"

#Pre-compute example pools once
sign_examples = {
    sign: (
        df[df["sign"] == sign]["horoscope"]
        .sample(6)  #changed to 1 for test, use 6 for actual run
        .str[:600]  #changed to 100 for test, use 600 for actual run
        .tolist()
    )
    for sign in signs
}

#Prompt builder
def build_prompt(sign, date):
    examples = random.sample(sign_examples[sign], 2)
    return f"""Write a daily horoscope for {sign} for {date.strftime('%B %d, %Y')}.
Match the tone and style of the examples.

Example 1:
{examples[0]}

Example 2:
{examples[1]}

Write a new horoscope of about 80-120 words."""

#Generation function
def generate_horoscope(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = ollama.chat(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                options={
                    "temperature": 0.8,
                    "num_predict": 200,    # equivalent to max_tokens
                }
            )
            return response.message.content.strip()
        except Exception as e:
            wait = 2 ** attempt * 2
            print(f"Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Generation failed after retries")

#worker function
if os.path.exists(CHECKPOINT_FILE):
    existing = pd.read_csv(CHECKPOINT_FILE)
    done = set(zip(existing["date"].astype(str), existing["sign"]))
    print(f"Resuming: {len(existing)} already done")
else:
    done = set()


def generate_one(date, sign):
    if (str(date.date()), sign) in done:
        return None
    prompt = build_prompt(sign, date)
    text = generate_horoscope(prompt)
    return {"date": date.date(), "sign": sign, "horoscope": text}


#Generation loop with checkpointing, in case the job fails

rows = []
CHECKPOINT_FILE = "horoscopes_ollama.csv"

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {
        executor.submit(generate_one, date, sign): (date, sign)
        for date in dates
        for sign in signs
    }
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result:
            rows.append(result)
            if len(rows) % 50 == 0:
                pd.DataFrame(rows).to_csv(
                    CHECKPOINT_FILE,
                    mode="a",
                    header=not os.path.exists(CHECKPOINT_FILE),
                    index=False
                )
                rows = []  

# Save any remaining rows after loop finishes
if rows:
    pd.DataFrame(rows).to_csv(
        CHECKPOINT_FILE,
        mode="a",
        header=not os.path.exists(CHECKPOINT_FILE),
        index=False
    )